# 직각 주차 디버깅 노트북

초음파 센서 5개만으로 수직(직각) 주차 미션을 수행하는 로직의 개발/디버깅용 노트북입니다.
경기 당일에는 규정상 `.py`로 변환해서 사용해야 합니다.

---

## 📌 실행 방법

노트북이 세 파트로 나뉩니다.

| 파트 | 내용 | 실행 |
|---|---|---|
| **A. 정의** | 임포트, 클래스, CONFIG | **A-1부터 A-9까지 순서대로 전부 실행** |
| **B. 테스트 / 캘리브레이션** | 모터·SPI·초음파 확인, 상수 실측 | 필요한 것만 골라서 |
| **C. 실행** | 주차 상태머신 구동 | |

**파트 A의 셀은 정의만 하고 즉시 끝납니다.** 무한 루프가 도는 셀은 전부 파트 B와 C에만 있습니다.
따라서 **A를 통째로 실행한 뒤 B에서 원하는 테스트만 골라 돌리는 것**이 기본 사용법입니다.

```
A (전부)  →  B (원하는 것만)  →  C (실행)
```

> ⚠️ 커널을 재시작하면 파트 A를 처음부터 다시 실행해야 합니다.

---

## 센서 배치

가이드라인의 12개 센서 번호 중 5개를 사용합니다.

| 이름 | 위치 | 역할 |
|---|---|---|
| `S0` | 우측 후측면 | 공간 탐색, 평행 정렬, 중앙 정렬, 출차 판단 |
| `S2` | 우측 전측면 | 공간 탐색, 좌회전 종료, 평행 정렬 |
| `S8` | 좌측 후측면 | 중앙 정렬, 후진 진입 판단, 출차 판단 |
| `S9` | 좌측 후방 코너 (45도) | 정렬 후진 종료, 후방 충돌 방지 |
| `S11` | 우측 후방 코너 (45도) | 정렬 후진 종료, 후방 충돌 방지 |


## 핀 배선 (`pwm_pins.xdc` 기준)

| 센서 | trig | echo |
|---|---|---|
| 0 `S0` | J1[7] (F6) | J1[8] (G5) |
| 1 `S2` | J1[9] (A6) | J1[10] (A7) |
| 2 `S8` | J2[3] (E5) | J2[4] (D6) |
| 3 `S9` | J2[7] (D5) | J2[8] (C7) |
| 4 `S11` | J2[9] (B6) | J2[10] (C5) |

## 규정에서 온 제약

- **후진으로** 주차 → 3~5초 정차 → 출차 → OUT 라인을 앞바퀴가 통과해야 성공
- **차동 회전 금지**: 좌우 후륜 속도는 항상 동일해야 함 (`set_speed()` 하나로만 제어)
- 출발 위치 4곳 / 장애물 배치 2가지가 **당일 추첨** → 거리·시간 하드코딩 금지, 센서 이벤트 기반
- 제한시간 4분, 충돌 회당 -20점, 차선 침범 -5점

---
# 파트 A — 정의

**A-1 ~ A-9를 순서대로 전부 실행하세요.** 모두 정의만 하고 즉시 끝나며, 무한 루프가 없습니다.

## A-1. 임포트 & 오버레이 로드

In [1]:
import time
import sys
import math
from collections import deque
from IPython.display import clear_output

from pynq import Overlay, MMIO
import spidev

# dpu.bit / dpu.hwh 가 같은 폴더에 있어야 합니다.
BITSTREAM = "../dpu_ultrasonic_cleaned/dpu.bit"
overlay = Overlay(BITSTREAM)

print(f"loaded: {BITSTREAM}\n")
print("사용 가능한 IP 목록:")
for name, info in overlay.ip_dict.items():
    addr = info.get("phys_addr")
    rng = info.get("addr_range")
    if addr is None:
        print(f"  {name:32s} (주소 매핑 없음)")
    else:
        rng_s = f"0x{rng:X}" if rng is not None else "?"
        print(f"  {name:32s} 0x{addr:010X}  range {rng_s}")

loaded: ../dpu_ultrasonic_cleaned/dpu.bit

사용 가능한 IP 목록:
  PWM_IP_0                         0x00A0000000  range 0x10000
  PWM_IP_1                         0x00A0010000  range 0x10000
  PWM_IP_2                         0x00A0020000  range 0x10000
  PWM_IP_3                         0x00A0030000  range 0x10000
  PWM_IP_4                         0x00A0040000  range 0x10000
  PWM_IP_5                         0x00A0050000  range 0x10000
  HC_SR04_IP_1                     0x00A0070000  range 0x10000
  DPUCZDX8G_1                      0x00B0000000  range 0x1000
  axi_intc_0                       0x0080000000  range 0x10000
  ps_e                             (주소 매핑 없음)


## A-2. 모터 정의

주소는 A-1의 IP 목록과 대조해서 확인하세요.

모터 역할은 주행 노트북의 `left()` / `right()` 함수 동작 기준입니다
(딕셔너리 주석에 좌우가 뒤바뀐 오타가 있어 그쪽은 신뢰하지 않았습니다).
**B-1에서 실제로 확인하세요.**

In [2]:
MOTOR_BASE = {
    "rear_right_back":  0x00A0000000,   # motor_0
    "steer_left":       0x00A0010000,   # motor_1
    "rear_left_back":   0x00A0020000,   # motor_2
    "rear_left_front":  0x00A0030000,   # motor_3
    "rear_right_front": 0x00A0040000,   # motor_4
    "steer_right":      0x00A0050000,   # motor_5
}
MOTOR_ADDR_RANGE = 0x10000

# PWM IP 레지스터 맵 (PWM_rev00.v 기준)
PWM_REG_SIZE  = 0x00   # 주기
PWM_REG_DUTY  = 0x04   # 듀티 사이클
PWM_REG_VALID = 0x08   # 모터 활성화

# 유아용 전동차 모터 한 주기 = 2ms, 클럭 3.333ns -> 600600 카운트
PWM_SIZE = 600600

motors = {name: MMIO(addr, MOTOR_ADDR_RANGE) for name, addr in MOTOR_BASE.items()}

def motor_init_all():
    """모든 모터: 주기 설정, 듀티 0, valid=0 (정지)."""
    for m in motors.values():
        m.write(PWM_REG_SIZE, PWM_SIZE)
        m.write(PWM_REG_DUTY, 0)
        m.write(PWM_REG_VALID, 0)

def motor_all_off():
    for m in motors.values():
        m.write(PWM_REG_VALID, 0)

def duty_to_reg(percent):
    """듀티 % -> 레지스터 값 (0~100 클램프)."""
    percent = max(0.0, min(100.0, percent))
    return int(PWM_SIZE * (percent / 100.0))

motor_init_all()
print("모터 초기화 완료 (전부 정지 상태)")

모터 초기화 완료 (전부 정지 상태)


## A-3. SPI (조향 가변저항) 정의

`read_adc()` 함수만 정의합니다. `VehicleController.read_steer()`가 이 함수를 사용하므로
**조향을 쓰지 않더라도 반드시 실행해야 합니다.**

실제 ADC 값 범위를 눈으로 확인하는 것은 B-2에서 합니다.

In [3]:
spi0 = spidev.SpiDev()
spi0.open(0, 0)
spi0.max_speed_hz = 1_000_000
spi0.mode = 0b00

def read_adc(spi=None):
    """Pmod AD1에서 12bit ADC 값 읽기."""
    r = (spi or spi0).xfer2([0x00, 0x00])
    return ((r[0] & 0x0F) << 8) | r[1]

print(f"SPI 초기화 완료. 현재 ADC = {read_adc()}")

SPI 초기화 완료. 현재 ADC = 2400


## A-4. 초음파 센서 클래스 정의

`MedianFilter`, `UltrasonicChannel`, `UltrasonicArray`를 정의하고 `sonic` MMIO 객체를 만듭니다.

- `timeout`은 `MAX_VALID_CM`(400cm)으로 치환 → "감지 범위 내 반사면 없음"
- median-5는 튀는 값(유령 에코)은 걸러내면서 계단 변화는 그대로 통과
- 같은 측정값을 중복으로 창에 넣지 않도록 tick 변화를 감지

In [4]:
SONIC_IP_NAME = "HC_SR04_IP_1"

SONIC_REG_CONTROL = 0x00     # bit0 = 연속 측정 enable
SONIC_REG_STATUS  = 0x04     # {timeout[20:16], busy[12:8], valid[4:0]}
SONIC_DIST_OFFSETS = (0x1C, 0x20, 0x24, 0x28, 0x2C)

NUM_SENSORS = 5
SONIC_CLOCK_HZ = 300_000_000
SPEED_OF_SOUND_CM_S = 34_300.0
MIN_VALID_CM = 2.0
MAX_VALID_CM = 400.0

if SONIC_IP_NAME not in overlay.ip_dict:
    raise RuntimeError(f"{SONIC_IP_NAME} not found. available: {list(overlay.ip_dict)}")

_si = overlay.ip_dict[SONIC_IP_NAME]
sonic = MMIO(_si["phys_addr"], _si["addr_range"])
print(f"{SONIC_IP_NAME} @ 0x{_si['phys_addr']:08X}")


class MedianFilter:
    def __init__(self, window=5):
        self.window = window
        self.buf = deque(maxlen=window)

    def update(self, value):
        self.buf.append(value)
        return sorted(self.buf)[len(self.buf) // 2]

    @property
    def filled(self):
        return len(self.buf) == self.window

    def reset(self):
        self.buf.clear()


class UltrasonicChannel:
    """1채널. 새 측정일 때만 median 창을 갱신."""

    def __init__(self, index, median_window=5,
                 max_hold_s=0.5, timeout_interval_s=0.15):
        self.index = index
        self.median = MedianFilter(median_window)
        self.max_hold_s = max_hold_s
        self.timeout_interval_s = timeout_interval_s

        self.raw_cm = None
        self.filtered_cm = None
        self.state = "waiting"
        self.is_new_sample = False
        self.sample_count = 0
        self.last_update_time = None
        self.update_period = None
        self._last_ticks = None

    def _push(self, cm, now):
        cm = min(max(cm, MIN_VALID_CM), MAX_VALID_CM)
        self.raw_cm = cm
        self.filtered_cm = self.median.update(cm)
        self.is_new_sample = True
        self.sample_count += 1
        if self.last_update_time is not None:
            self.update_period = now - self.last_update_time
        self.last_update_time = now

    def update(self, ticks, is_valid, is_busy, is_timeout, now):
        self.is_new_sample = False

        if is_timeout:
            self.state = "timeout"
            # timeout 지속 중에는 바퀴마다 "감지 없음"이 새로 측정되는 것이므로
            # 주기적으로 계속 밀어넣어야 한다.
            if (self.last_update_time is None
                    or now - self.last_update_time >= self.timeout_interval_s):
                self._last_ticks = None
                self._push(MAX_VALID_CM, now)
            return self.filtered_cm

        if is_valid:
            self.state = "valid"
            stale = (self.last_update_time is not None
                     and now - self.last_update_time > self.max_hold_s)
            if ticks != self._last_ticks or stale:
                self._last_ticks = ticks
                self._push(ticks / SONIC_CLOCK_HZ * SPEED_OF_SOUND_CM_S / 2.0, now)
            return self.filtered_cm

        self.state = "measuring" if is_busy else "waiting"
        return self.filtered_cm

    @property
    def reliable(self):
        return self.median.filled


class UltrasonicArray:
    def __init__(self, mmio, median_window=5):
        self.mmio = mmio
        self.channels = [UltrasonicChannel(i, median_window)
                         for i in range(NUM_SENSORS)]

    def start(self):
        self.mmio.write(SONIC_REG_CONTROL, 0x1)

    def stop(self):
        self.mmio.write(SONIC_REG_CONTROL, 0x0)

    def poll(self):
        now = time.monotonic()
        status = self.mmio.read(SONIC_REG_STATUS)
        valid = status & 0x1F
        busy = (status >> 8) & 0x1F
        timeout = (status >> 16) & 0x1F
        for i, off in enumerate(SONIC_DIST_OFFSETS):
            mask = 1 << i
            self.channels[i].update(
                self.mmio.read(off),
                is_valid=bool(valid & mask),
                is_busy=bool(busy & mask),
                is_timeout=bool(timeout & mask),
                now=now,
            )

    def report(self):
        out = []
        for ch in self.channels:
            f = f"{ch.filtered_cm:6.1f}" if ch.filtered_cm is not None else "  ----"
            mark = "*" if ch.is_new_sample else " "
            warm = "" if ch.reliable else " (warming up)"
            out.append(f"{mark} ch{ch.index} [{ch.state:9s}] {f} cm{warm}")
        return "\n".join(out)


print("초음파 클래스 정의 완료")

HC_SR04_IP_1 @ 0xA0070000
초음파 클래스 정의 완료


## A-5. CONFIG — 튜닝 상수 전부

**현장에서 고치는 값은 전부 여기에만 있습니다.** 다른 셀은 이 값을 참조만 합니다.

`# TUNE` 표시가 붙은 것은 실측으로 채워야 하는 값입니다. 파트 B에서 하나씩 구할 수 있습니다.

In [5]:
CFG = {
    # ---------- 센서 매핑 (가이드 번호 -> 하드웨어 채널 0~4) ----------
    "SENSOR_CH": {                          # TUNE: 실제 배선에 맞게
        "S0":  0,   # 우측 후측면
        "S2":  1,   # 우측 전측면
        "S8":  2,   # 좌측 후측면
        "S9":  3,   # 좌측 후방 코너 (45도)
        "S11": 4,   # 우측 후방 코너 (45도)
    },

    # ---------- 밴드(구간) 임계값 ----------
    # 경계 4개 -> 밴드 5개: 0=아주 가까움 1=가까움 2=적당함 3=멈 4=아주 멈
    "BAND_EDGES_CM": [30.0, 60.0, 120.0, 200.0],   # TUNE
    "BAND_HYST_CM": 4.0,        # 경계에서 딸랑거림 방지용 히스테리시스

    # 장애물이 "감지됐다"고 볼 최대 밴드 (이 값 이하면 감지)
    "DETECT_BAND": 2,           # 적당함 이하 = 감지
    # 센서별 예외. S9·S11은 45도로 옆 차량을 비스듬히 보기 때문에
    # 같은 상황에서도 측면 센서보다 거리가 길게 나온다.
    "DETECT_BAND_OVERRIDE": {"S9": 3, "S11": 3},   # TUNE
    # 완전히 비었다고 볼 밴드
    "CLEAR_BAND": 4,            # 아주 멈

    # ---------- 조향 ----------
    "ADC_MOST_LEFT":  1848,     # TUNE: 4번 셀에서 실측
    "ADC_MOST_RIGHT":  984,     # TUNE
    "STEER_RANGE": 7.0,         # 매핑 범위 (-7 ~ +7). +가 우조향
    "STEER_DUTY": 70.0,         # 조향 모터 고정 듀티 %      # TUNE
    "STEER_DEADBAND": 0.5,      # 이 안에 들어오면 조향 정지
    "STEER_FULL": 7.0,          # 최대 조향에 쓸 목표값
    "STEER_ALIGN_STEP": 2.0,    # 정렬 후진에서 쓸 조향 크기  # TUNE (2~4)
    "STEER_SETTLE_TIMEOUT": 3.0,  # 목표 조향 도달 대기 최대 시간(초)

    # ---------- 주행 속도 (듀티 %) ----------
    "SPEED_FORWARD": 40.0,      # TUNE: 8번에서 최소 구동 듀티 확인 후
    "SPEED_REVERSE": 35.0,      # TUNE
    "SPEED_ALIGN":   30.0,      # 정렬 후진은 더 느리게  # TUNE
    "SPEED_EXIT":    40.0,      # TUNE

    # ---------- 정렬 제어 (5단계: S0-S2 각도만) ----------
    # err = S0 - S2  (S0가 더 멀면 뒤가 벌어진 것 -> 각도 보정)
    "ALIGN_DEADBAND_CM": 2.0,   # TUNE: 이 안이면 조향 중립
    "ALIGN_SIGN": +1,           # TUNE: 조향 방향이 반대로 먹으면 -1로
    "ALIGN_MIN_SECONDS": 2.0,   # 이 시간 전에는 종료 판정을 하지 않음  # TUNE

    # ---------- 안전 ----------
    "REAR_STOP_CM": 10.0,       # S9/S11이 이보다 가까우면 후진 즉시 중단  # TUNE

    # ---------- 정차 / 출차 ----------
    "HOLD_SECONDS": 4.0,        # 규정 3~5초
    "EXIT_TURN_SECONDS": 2.5,   # TUNE: 90도 우회전에 걸리는 시간
    "EXIT_STRAIGHT_SECONDS": 20.0,  # TUNE: OUT 라인까지 직진

    # ---------- 타임아웃 (초) ----------
    "TIMEOUT": {
        "SEARCH_GAP":     60.0,
        "SWING_LEFT":     20.0,
        "REVERSE_RIGHT":  20.0,
        "ALIGN_REVERSE":  40.0,
        "EXIT_FORWARD":   20.0,
    },
    "MISSION_TIMEOUT": 230.0,   # 규정 4분보다 약간 짧게

    # ---------- 루프 ----------
    "LOOP_INTERVAL": 0.05,
}

BAND_NAMES = ["아주 가까움", "가까움", "적당함", "멈", "아주 멈"]

print("CONFIG 로드 완료")
for k in ("SPEED_FORWARD", "SPEED_ALIGN", "STEER_ALIGN_STEP", "BAND_EDGES_CM"):
    print(f"  {k:20s} = {CFG[k]}")

CONFIG 로드 완료
  SPEED_FORWARD        = 40.0
  SPEED_ALIGN          = 30.0
  STEER_ALIGN_STEP     = 2.0
  BAND_EDGES_CM        = [30.0, 60.0, 120.0, 200.0]


## A-6. 센서 밴드 (Bander / SensorView)

### Bander
연속값(cm)을 5단계 밴드로 바꿉니다. 경계에서 값이 딸랑거리지 않도록
히스테리시스를 적용합니다 — 올라갈 때는 `경계 + hyst`를 넘어야 하고,
내려올 때는 `경계 - hyst` 아래로 떨어져야 합니다.

**밴드는 이벤트 판단에만, 정렬 제어에는 연속값(cm)을 씁니다.**
정렬은 몇 cm 단위의 차이를 봐야 하는데 밴드로 뭉개면 그 정보가 사라집니다.

In [6]:
class Bander:
    """연속 거리값 -> 히스테리시스가 적용된 밴드 인덱스."""

    def __init__(self, edges, hyst):
        self.edges = list(edges)
        self.hyst = hyst
        self.idx = None

    def _raw(self, cm):
        i = 0
        while i < len(self.edges) and cm > self.edges[i]:
            i += 1
        return i

    def update(self, cm):
        if cm is None:
            return self.idx
        if self.idx is None:
            self.idx = self._raw(cm)
            return self.idx
        i = self.idx
        while i < len(self.edges) and cm > self.edges[i] + self.hyst:
            i += 1
        while i > 0 and cm < self.edges[i - 1] - self.hyst:
            i -= 1
        self.idx = i
        return i


class SensorView:
    """UltrasonicArray를 S0/S2/S8/S9/S11 이름으로 보고, 밴드도 함께 제공."""

    def __init__(self, array, cfg):
        self.array = array
        self.cfg = cfg
        self.ch_of = cfg["SENSOR_CH"]
        self.banders = {
            name: Bander(cfg["BAND_EDGES_CM"], cfg["BAND_HYST_CM"])
            for name in self.ch_of
        }
        self.bands = {name: None for name in self.ch_of}

    def poll(self):
        self.array.poll()
        for name, ch in self.ch_of.items():
            cm = self.array.channels[ch].filtered_cm
            self.bands[name] = self.banders[name].update(cm)

    def cm(self, name):
        """필터링된 연속값(cm). 아직 없으면 None."""
        return self.array.channels[self.ch_of[name]].filtered_cm

    def band(self, name):
        return self.bands[name]

    def detect_limit(self, name):
        return self.cfg["DETECT_BAND_OVERRIDE"].get(name, self.cfg["DETECT_BAND"])

    def detected(self, name):
        b = self.bands[name]
        return b is not None and b <= self.detect_limit(name)

    def clear(self, name):
        b = self.bands[name]
        return b is not None and b >= self.cfg["CLEAR_BAND"]

    def ready(self):
        """모든 센서의 필터 창이 찼는지."""
        return all(self.array.channels[ch].reliable for ch in self.ch_of.values())

    def report(self):
        out = []
        for name in ("S0", "S2", "S8", "S9", "S11"):
            cm = self.cm(name)
            b = self.band(name)
            cms = f"{cm:6.1f}" if cm is not None else "  ----"
            bn = BAND_NAMES[b] if b is not None else "?"
            tag = " <감지>" if self.detected(name) else (" <빔>" if self.clear(name) else "")
            out.append(f"  {name:4s} {cms} cm  [{bn}]{tag}")
        return "\n".join(out)


print("Bander / SensorView 정의 완료")

Bander / SensorView 정의 완료


## A-7. 차량 제어 (VehicleController)

- `set_speed(v)` : 좌우 후륜을 **항상 같은 값**으로 구동 (규정상 차동 회전 금지)
- `steer_update(target)` : 논블로킹 조향. 매 루프마다 호출
- `steer_to(target)` : 블로킹 조향. 상태 진입 시 바퀴를 미리 꺾어둘 때 사용

In [7]:
class VehicleController:
    """모터/조향 저수준 제어. 차동 회전 금지 규정 때문에 좌우 속도는 항상 동일."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.motors = motors
        self._speed = 0.0
        self._steer_dir = 0          # -1 좌, 0 정지, +1 우
        motor_init_all()

    # ---------- 조향 ----------
    def read_steer(self):
        """현재 조향값을 -STEER_RANGE ~ +STEER_RANGE 로 반환. +가 우조향."""
        adc = read_adc()
        lo = self.cfg["ADC_MOST_RIGHT"]
        hi = self.cfg["ADC_MOST_LEFT"]
        r = self.cfg["STEER_RANGE"]
        if adc <= lo:
            return r
        if adc >= hi:
            return -r
        return (hi - adc) * (2 * r) / (hi - lo) - r

    def _steer_drive(self, direction):
        """direction: -1 좌, 0 정지, +1 우."""
        if direction == self._steer_dir:
            return
        self._steer_dir = direction
        duty = duty_to_reg(self.cfg["STEER_DUTY"])
        if direction == 0:
            self.motors["steer_left"].write(PWM_REG_VALID, 0)
            self.motors["steer_right"].write(PWM_REG_VALID, 0)
        elif direction > 0:
            self.motors["steer_left"].write(PWM_REG_VALID, 0)
            self.motors["steer_right"].write(PWM_REG_DUTY, duty)
            self.motors["steer_right"].write(PWM_REG_VALID, 1)
        else:
            self.motors["steer_right"].write(PWM_REG_VALID, 0)
            self.motors["steer_left"].write(PWM_REG_DUTY, duty)
            self.motors["steer_left"].write(PWM_REG_VALID, 1)

    def steer_update(self, target):
        """논블로킹 조향 1스텝. 목표 도달 여부를 반환."""
        cur = self.read_steer()
        err = target - cur
        if abs(err) <= self.cfg["STEER_DEADBAND"]:
            self._steer_drive(0)
            return True
        self._steer_drive(1 if err > 0 else -1)
        return False

    def steer_to(self, target, timeout=None):
        """블로킹 조향. 상태 진입 시 바퀴를 미리 꺾어둘 때 사용."""
        timeout = timeout or self.cfg["STEER_SETTLE_TIMEOUT"]
        t0 = time.monotonic()
        while time.monotonic() - t0 < timeout:
            if self.steer_update(target):
                return True
            time.sleep(0.02)
        self._steer_drive(0)
        return False

    def steer_stop(self):
        self._steer_drive(0)

    # ---------- 주행 ----------
    def set_speed(self, speed):
        """좌우 후륜 동시 구동. speed>0 전진, <0 후진, 0 정지."""
        self._speed = speed
        duty = duty_to_reg(abs(speed))
        for side in ("left", "right"):
            self.motors[f"rear_{side}_front"].write(PWM_REG_DUTY, duty)
            self.motors[f"rear_{side}_back"].write(PWM_REG_DUTY, duty)

        if speed > 0:
            for side in ("left", "right"):
                self.motors[f"rear_{side}_back"].write(PWM_REG_VALID, 0)
                self.motors[f"rear_{side}_front"].write(PWM_REG_VALID, 1)
        elif speed < 0:
            for side in ("left", "right"):
                self.motors[f"rear_{side}_front"].write(PWM_REG_VALID, 0)
                self.motors[f"rear_{side}_back"].write(PWM_REG_VALID, 1)
        else:
            for side in ("left", "right"):
                self.motors[f"rear_{side}_front"].write(PWM_REG_VALID, 0)
                self.motors[f"rear_{side}_back"].write(PWM_REG_VALID, 0)

    def stop(self):
        self.set_speed(0)

    def all_off(self):
        self.set_speed(0)
        self.steer_stop()
        motor_all_off()


print("VehicleController 정의 완료")

VehicleController 정의 완료


## A-8. 주차 상태머신

### 상태 흐름

```
INIT
 -> SEARCH_GAP      직진.  S2 감지 -> S2·S0 모두 비면 정지
 -> SWING_LEFT      최대 좌조향 전진.  S2 재감지 -> 다시 비면 정지
 -> REVERSE_RIGHT   최대 우조향 후진.  S0·S8 모두 감지되면 정지
 -> ALIGN_REVERSE   캐스케이드 정렬 저속 후진.  S9·S11 감지 -> 모두 비면 정지
 -> HOLD            3~5초 정차 (규정)
 -> EXIT_FORWARD    직진.  S0·S8 감지 -> 모두 비면 정지
 -> EXIT_TURN       최대 우조향 전진 (시간 기반 90도)
 -> EXIT_STRAIGHT   직진 (OUT 라인 통과)
 -> DONE
```

어느 상태에서든 타임아웃이나 후방 근접이 발생하면 `ABORT`로 빠져 전부 정지합니다.

### "감지 → 해제" 2단계 판정

`SEARCH_GAP`, `SWING_LEFT`, `ALIGN_REVERSE`, `EXIT_FORWARD`의 종료 조건은
단순히 "센서가 비었나"가 아니라 **"한 번 감지한 뒤에 비었나"** 입니다.

median 필터는 상태가 바뀌어도 이전 구간의 값을 창에 들고 있습니다.
그래서 "비었다"만 보면, 상태에 진입하자마자 직전 구간의 잔상 때문에
조건이 즉시 만족되어 그 단계를 통째로 건너뛰는 일이 생깁니다.
특히 `ALIGN_REVERSE`는 진입 시점에 S9·S11이 아직 옆 차량을 못 봤을 수 있어서,
2단계 판정이 없으면 **깊이가 전혀 확보되지 않은 채 주차가 끝나버립니다.**

### 정렬 캐스케이드

조향 입력은 하나인데 맞춰야 할 목표가 둘(각도, 좌우 위치)이라 동시에는 못 맞춥니다.
그래서 2단으로 나눕니다.

```
바깥 루프:  offset = K_CENTER x (S0 - S8)        중앙 오차 -> 목표 각도 오프셋
안쪽 루프:  err = (S0 - S2) - offset             각도 오차
            |err| <= deadband  -> 조향 0
            그 외              -> 조향 ±STEER_ALIGN_STEP
```

> ⚠️ `ALIGN_SIGN`은 실측으로 정해야 합니다. 후진 중에는 조향 방향의 효과가
> 전진과 반대라, 처음 돌렸을 때 오차가 줄어들지 않고 커지면 `-1`로 뒤집으세요.

In [8]:
class ParkingStateMachine:

    STATES = ("INIT", "SEARCH_GAP", "SWING_LEFT", "REVERSE_RIGHT",
              "ALIGN_REVERSE", "HOLD", "EXIT_FORWARD", "EXIT_TURN",
              "EXIT_STRAIGHT", "DONE", "ABORT")

    def __init__(self, car, view, cfg):
        self.car = car
        self.view = view
        self.cfg = cfg
        self.state = "INIT"
        self.state_t0 = time.monotonic()
        self.mission_t0 = time.monotonic()
        self.log = []
        self.abort_reason = None
        self._flag = False        # 상태별 1회성 서브페이즈 플래그
        self._steer_target = 0.0

    # ---------------- 공통 ----------------
    def elapsed(self):
        return time.monotonic() - self.state_t0

    def mission_elapsed(self):
        return time.monotonic() - self.mission_t0

    def goto(self, new_state):
        self.log.append({
            "t": round(self.mission_elapsed(), 2),
            "from": self.state,
            "to": new_state,
            "sensors": {n: self.view.cm(n) for n in ("S0", "S2", "S8", "S9", "S11")},
        })
        self.state = new_state
        self.state_t0 = time.monotonic()
        self._flag = False
        self.on_enter()

    def abort(self, reason):
        self.abort_reason = reason
        self.car.all_off()
        self.goto("ABORT")

    def rear_too_close(self):
        """후방 코너 센서가 위험 거리 안에 들어왔는지."""
        limit = self.cfg["REAR_STOP_CM"]
        for name in ("S9", "S11"):
            cm = self.view.cm(name)
            if cm is not None and cm < limit:
                return name
        return None

    def check_timeout(self):
        t = self.cfg["TIMEOUT"].get(self.state)
        if t is not None and self.elapsed() > t:
            self.abort(f"{self.state} 타임아웃 ({t}s)")
            return True
        if self.mission_elapsed() > self.cfg["MISSION_TIMEOUT"]:
            self.abort("미션 전체 타임아웃")
            return True
        return False

    # ---------------- 상태 진입 ----------------
    def on_enter(self):
        c, cfg = self.car, self.cfg
        s = self.state

        if s == "SEARCH_GAP":
            c.stop()
            c.steer_to(0.0)
            self._steer_target = 0.0
            c.set_speed(+cfg["SPEED_FORWARD"])

        elif s == "SWING_LEFT":
            c.stop()
            c.steer_to(-cfg["STEER_FULL"])
            self._steer_target = -cfg["STEER_FULL"]
            c.set_speed(+cfg["SPEED_FORWARD"])

        elif s == "REVERSE_RIGHT":
            c.stop()
            c.steer_to(+cfg["STEER_FULL"])
            self._steer_target = +cfg["STEER_FULL"]
            c.set_speed(-cfg["SPEED_REVERSE"])

        elif s == "ALIGN_REVERSE":
            c.stop()
            c.steer_to(0.0)
            self._steer_target = 0.0
            c.set_speed(-cfg["SPEED_ALIGN"])

        elif s == "HOLD":
            c.stop()
            c.steer_to(0.0)
            self._steer_target = 0.0

        elif s == "EXIT_FORWARD":
            c.stop()
            c.steer_to(0.0)
            self._steer_target = 0.0
            c.set_speed(+cfg["SPEED_EXIT"])

        elif s == "EXIT_TURN":
            c.stop()
            c.steer_to(+cfg["STEER_FULL"])
            self._steer_target = +cfg["STEER_FULL"]
            c.set_speed(+cfg["SPEED_EXIT"])

        elif s == "EXIT_STRAIGHT":
            c.stop()
            c.steer_to(0.0)
            self._steer_target = 0.0
            c.set_speed(+cfg["SPEED_EXIT"])

        elif s in ("DONE", "ABORT"):
            c.all_off()

    # ---------------- 정렬 (5단계: S0-S2 각도만) ----------------
    def align_steer_target(self):
        cfg = self.cfg
        s0, s2 = self.view.cm("S0"), self.view.cm("S2")
        if s0 is None or s2 is None:
            return 0.0, None

        # 5단계: S0(우측 후측면)와 S2(우측 전측면)를 비교해 각도(평행)만 맞춘다.
        # 6단계(S0-S8 중앙 정렬)는 제거했으므로 좌우 위치는 보정하지 않는다.
        err = s0 - s2

        if abs(err) <= cfg["ALIGN_DEADBAND_CM"]:
            target = 0.0
        else:
            direction = 1.0 if err > 0 else -1.0
            target = cfg["ALIGN_SIGN"] * direction * cfg["STEER_ALIGN_STEP"]
        return target, err

    # ---------------- 상태 갱신 ----------------
    def update(self):
        v, cfg = self.view, self.cfg
        s = self.state

        if s in ("DONE", "ABORT"):
            return

        if self.check_timeout():
            return

        # 후진 중 안전 검사
        if s in ("REVERSE_RIGHT", "ALIGN_REVERSE"):
            hit = self.rear_too_close()
            if hit is not None:
                self.abort(f"{hit} 후방 근접 ({v.cm(hit):.1f}cm)")
                return

        if s == "INIT":
            if v.ready():
                self.goto("SEARCH_GAP")
            return

        if s == "SEARCH_GAP":
            # 1단계 직진 + 2단계 공간 감지
            if not self._flag:
                if v.detected("S2"):
                    self._flag = True          # 첫 장애물 차량을 봤다
            else:
                if v.clear("S2") and v.clear("S0"):
                    self.car.stop()
                    self.goto("SWING_LEFT")
            return

        if s == "SWING_LEFT":
            # 3단계: 최대 좌조향 전진. 원본대로 S0(우측 후측면) 기준.
            # S0가 다시 감지됐다가 풀리면(감지 -> 해제 2단계 래치) 정지.
            self.car.steer_update(self._steer_target)
            if not self._flag:
                if v.detected("S0"):
                    self._flag = True
            else:
                if v.clear("S0"):
                    self.car.stop()
                    self.goto("REVERSE_RIGHT")
            return

        if s == "REVERSE_RIGHT":
            # 4단계: 최대 우조향 후진. 원본대로 S2(우측 전측면)가 우측 장애물
            # 차량에 가까워지면(감지) 정지. 진입 직후 S2 잔상/근접으로 즉시
            # 오종료되는 것을 막기 위해 "먼저 S2가 해제된 뒤 감지"(해제 -> 감지)
            # 2단계 가드를 둔다.
            self.car.steer_update(self._steer_target)
            if not self._flag:
                if v.clear("S2"):
                    self._flag = True          # S2가 비었음을 확인
            else:
                if v.detected("S2"):
                    self.car.stop()
                    self.goto("ALIGN_REVERSE")
            return

        if s == "ALIGN_REVERSE":
            # 5단계: S0-S2 각도 정렬 저속 후진 + 7단계 종료 판정.
            target, _ = self.align_steer_target()
            self._steer_target = target
            self.car.steer_update(target)

            # 종료 판정도 "감지 -> 해제" 2단계로 한다.
            # 진입 직후에는 S9·S11이 아직 옆 차량을 못 봤을 수 있는데,
            # 그 상태의 clear를 바로 종료로 받으면 깊이가 부족한 채 끝난다.
            if not self._flag:
                if self.elapsed() >= cfg["ALIGN_MIN_SECONDS"] and (
                        v.detected("S9") or v.detected("S11")):
                    self._flag = True
            else:
                if v.clear("S9") and v.clear("S11"):
                    self.car.stop()
                    self.goto("HOLD")
            return

        if s == "HOLD":
            if self.elapsed() >= cfg["HOLD_SECONDS"]:
                self.goto("EXIT_FORWARD")
            return

        if s == "EXIT_FORWARD":
            self.car.steer_update(self._steer_target)
            # 뒤쪽 센서 두 개가 모두 풀려야 차 뒷부분까지 빠져나온 것.
            # 여기서도 먼저 감지 상태를 확인한 뒤 해제를 기다린다.
            if not self._flag:
                if v.detected("S0") or v.detected("S8"):
                    self._flag = True
            else:
                if v.clear("S0") and v.clear("S8"):
                    self.car.stop()
                    self.goto("EXIT_TURN")
            return

        if s == "EXIT_TURN":
            self.car.steer_update(self._steer_target)
            if self.elapsed() >= cfg["EXIT_TURN_SECONDS"]:
                self.car.stop()
                self.goto("EXIT_STRAIGHT")
            return

        if s == "EXIT_STRAIGHT":
            self.car.steer_update(self._steer_target)
            if self.elapsed() >= cfg["EXIT_STRAIGHT_SECONDS"]:
                self.car.stop()
                self.goto("DONE")
            return

    # ---------------- 표시 ----------------
    def report(self):
        lines = [
            f"상태: {self.state:14s}  경과 {self.elapsed():5.1f}s  "
            f"미션 {self.mission_elapsed():5.1f}s",
            f"조향: 현재 {self.car.read_steer():+5.2f}  목표 {self._steer_target:+5.2f}",
            self.view.report(),
        ]
        if self.state == "ALIGN_REVERSE":
            _, err = self.align_steer_target()
            s0, s2 = self.view.cm("S0"), self.view.cm("S2")
            if err is not None:
                lines.append(
                    f"  정렬: 각도차(S0-S2) {s0 - s2:+6.1f}  오차 {err:+6.1f}"
                )
        if self.abort_reason:
            lines.append(f"  중단 사유: {self.abort_reason}")
        return "\n".join(lines)


print("ParkingStateMachine 정의 완료")

ParkingStateMachine 정의 완료


## A-9. 정의 확인

파트 A가 빠짐없이 실행됐는지 점검합니다.
**여기서 `모든 정의 완료`가 뜨면 파트 B, C를 자유롭게 실행할 수 있습니다.**

In [9]:
_required = {
    "A-1 임포트/오버레이": ["overlay", "time", "clear_output"],
    "A-2 모터":            ["motors", "motor_init_all", "motor_all_off", "duty_to_reg"],
    "A-3 SPI":             ["read_adc"],
    "A-4 초음파":          ["sonic", "MedianFilter", "UltrasonicChannel", "UltrasonicArray"],
    "A-5 CONFIG":          ["CFG", "BAND_NAMES"],
    "A-6 밴드":            ["Bander", "SensorView"],
    "A-7 차량 제어":       ["VehicleController"],
    "A-8 상태머신":        ["ParkingStateMachine"],
}

missing = {}
for section, names in _required.items():
    lack = [n for n in names if n not in globals()]
    if lack:
        missing[section] = lack

if missing:
    print("❌ 아직 실행하지 않은 셀이 있습니다.\n")
    for section, lack in missing.items():
        print(f"  {section:22s} 누락: {', '.join(lack)}")
    print("\n해당 섹션의 셀을 실행하세요.")
else:
    print("✅ 모든 정의 완료. 파트 B / C를 실행할 수 있습니다.")
    print(f"\n   초음파 IP    : {SONIC_IP_NAME}")
    print(f"   모터 개수    : {len(motors)}")
    print(f"   현재 ADC     : {read_adc()}")
    print(f"   조향 범위 설정: {CFG['ADC_MOST_RIGHT']} (우) ~ {CFG['ADC_MOST_LEFT']} (좌)")

✅ 모든 정의 완료. 파트 B / C를 실행할 수 있습니다.

   초음파 IP    : HC_SR04_IP_1
   모터 개수    : 6
   현재 ADC     : 2400
   조향 범위 설정: 984 (우) ~ 1848 (좌)


---
# 파트 B — 테스트 / 캘리브레이션

**필요한 것만 골라서 실행하세요.** 파트 A가 전부 실행된 상태여야 합니다.

무한 루프가 도는 셀은 ⏹ 정지 버튼(`Ctrl+C`)으로 빠져나옵니다.
정지해도 파트 A의 정의는 그대로 남습니다.

| 셀 | 목적 | 종료 | 얻는 CONFIG 값 |
|---|---|---|---|
| B-1 | 모터 배선/방향 확인 | 즉시 | `MOTOR_BASE` 검증 |
| B-2 | 조향 ADC 범위 실측 | ⏹ 정지 | `ADC_MOST_LEFT/RIGHT` |
| B-3 | 초음파 5채널 원시값 | ⏹ 정지 | 채널 생존 확인 |
| B-4 | 센서별 거리/밴드 확인 | ⏹ 정지 | `BAND_EDGES_CM` |
| B-5 | 최소 구동 듀티 실측 | 자동 | `SPEED_*` |
| B-6 | 조향 도달 시간 실측 | 자동 | `STEER_SETTLE_TIMEOUT` |

## B-1. 모터 개별 테스트

배선과 방향을 확인합니다. 한 번에 하나씩만 켜지므로 안전합니다.
**차량을 들어올리거나 바퀴가 뜬 상태에서** 테스트하세요.

각 셀을 실행하고 실제로 도는 모터가 이름과 맞는지 확인한 뒤,
다르면 A-2의 `MOTOR_BASE` 주소 매핑을 고치세요.

⚠️ 마지막 정지 셀을 꼭 실행하세요.

In [ ]:
def motor_only(name, percent=60):
    """지정한 모터 하나만 켜고 나머지는 끔."""
    motor_all_off()
    if name is None:
        print("전부 정지")
        return
    m = motors[name]
    m.write(PWM_REG_DUTY, duty_to_reg(percent))
    m.write(PWM_REG_VALID, 1)
    print(f"{name} ON  (duty {percent}%)")

In [ ]:
motor_only("rear_right_front")    # 우측 후륜 - 전진

In [ ]:
motor_only("rear_right_back")    # 우측 후륜 - 후진

In [ ]:
motor_only("rear_left_front")    # 좌측 후륜 - 전진

In [ ]:
motor_only("rear_left_back")    # 좌측 후륜 - 후진

In [ ]:
motor_only("steer_left")    # 조향 - 좌

In [ ]:
motor_only("steer_right")    # 조향 - 우

In [ ]:
motor_only(None)    # 전부 정지

## B-2. 조향 ADC 범위 실측

핸들을 손으로 좌우 끝까지 돌려보며 ADC 값의 실제 범위를 확인합니다.
여기서 얻은 값을 A-5 CONFIG의 `ADC_MOST_LEFT` / `ADC_MOST_RIGHT`에 넣으세요.

⏹ 정지 버튼으로 종료합니다.

In [ ]:
lo, hi = 4096, 0
try:
    while True:
        v = read_adc()
        lo, hi = min(lo, v), max(hi, v)
        sys.stdout.write(f"\rADC: {v:4d}   관측 범위: {lo} ~ {hi}   ")
        sys.stdout.flush()
        time.sleep(0.05)
except KeyboardInterrupt:
    print(f"\n\n최종 관측 범위: {lo} ~ {hi}")
    print("핸들을 좌우 끝까지 돌렸을 때의 값을 CONFIG에 넣으세요.")

# 실측 기록용 메모
# 좌측:
# 우측:

## B-3. 초음파 5채널 원시값 모니터

하드웨어 채널 번호(ch0~ch4) 그대로 표시합니다. 5채널이 모두 살아있는지 확인하세요.

- `*` : 이번 poll에서 새 측정이 들어온 채널
- `(warming up)` : median 창이 아직 안 찬 상태

⏹ 정지 버튼으로 종료합니다.

In [10]:
# 5채널 실시간 모니터 (⏹ 정지 버튼으로 종료)
_arr = UltrasonicArray(sonic)
_arr.start()
try:
    while True:
        _arr.poll()
        clear_output(wait=True)
        print(_arr.report())
        time.sleep(0.05)
except KeyboardInterrupt:
    _arr.stop()
    print("정지")

* ch0 [valid    ]   54.4 cm
* ch1 [valid    ]   54.5 cm
  ch2 [timeout  ]  400.0 cm
  ch3 [valid    ]  400.0 cm
  ch4 [valid    ]    2.0 cm정지


## B-4. 센서별 거리 / 밴드 확인

`S0` / `S2` / `S8` / `S9` / `S11` 이름으로 보고, 밴드 판정 결과도 함께 표시합니다.

차량을 주차 공간 옆 여러 위치에 놓고 실행해서 다음 두 값을 확인하세요.

- **장애물 차량을 볼 때** 각 센서가 반환하는 거리
- **빈 공간을 볼 때** 각 센서가 반환하는 거리

`BAND_EDGES_CM`이 이 두 값을 확실히 갈라놓는지 확인합니다.
특히 **S9·S11은 45도로 비스듬히 보기 때문에 측면 센서보다 거리가 길게 나오므로**,
`DETECT_BAND_OVERRIDE`가 맞게 설정됐는지 여기서 확인하세요.

⏹ 정지 버튼으로 종료합니다.

In [ ]:
arr = UltrasonicArray(sonic)
view = SensorView(arr, CFG)
arr.start()

try:
    while True:
        view.poll()
        clear_output(wait=True)
        print(view.report())
        print(f"\n필터 준비됨: {view.ready()}")
        time.sleep(0.1)
except KeyboardInterrupt:
    arr.stop()
    print("정지")

## B-5. 최소 구동 듀티 찾기

듀티가 너무 낮으면 DC 모터가 아예 안 돕니다. **바닥에 내려놓고** 실행해서
차가 실제로 움직이기 시작하는 듀티를 확인하세요.
그 값에 여유를 더해 `SPEED_ALIGN` / `SPEED_FORWARD`에 넣습니다.

⚠️ 차량이 실제로 전진합니다. 앞쪽 공간을 확보하세요.

In [ ]:
car = VehicleController(CFG)

try:
    for pct in range(15, 61, 5):
        print(f"duty {pct}% ... 움직이는지 확인")
        car.set_speed(pct)
        time.sleep(1.5)
        car.stop()
        time.sleep(1.0)
finally:
    car.all_off()
print("완료. 확실히 움직이기 시작한 듀티 + 5~10%를 SPEED_ALIGN에 넣으세요.")

## B-6. 조향 범위 / 도달 시간 확인

최대 좌·우 조향까지 실제로 도달하는지, 얼마나 걸리는지 봅니다.
`STEER_SETTLE_TIMEOUT`을 여기서 나온 시간보다 넉넉하게 잡으세요.

**`실패(타임아웃)`가 뜨면** `ADC_MOST_LEFT/RIGHT` 값이 실제 범위와 안 맞거나
`STEER_DUTY`가 너무 낮은 것입니다.

In [ ]:
car = VehicleController(CFG)

for target, label in [(0.0, "중립"),
                      (-CFG["STEER_FULL"], "최대 좌"),
                      (0.0, "중립"),
                      (+CFG["STEER_FULL"], "최대 우"),
                      (0.0, "중립")]:
    t0 = time.monotonic()
    ok = car.steer_to(target, timeout=5.0)
    dt = time.monotonic() - t0
    print(f"{label:8s} 목표 {target:+5.1f} -> 현재 {car.read_steer():+5.2f}  "
          f"{dt:4.2f}s  {'OK' if ok else '실패(타임아웃)'}")

car.all_off()

---
# 파트 C — 실행

## C-1. 주차 상태머신 실행

⏹ 정지 버튼(`Ctrl+C`)을 누르면 언제든 즉시 모든 모터가 꺼집니다.
**처음 돌릴 때는 차량을 들어올린 상태로** 상태 전환만 확인하는 것을 권장합니다.

In [ ]:
arr = UltrasonicArray(sonic)
view = SensorView(arr, CFG)
car = VehicleController(CFG)
fsm = ParkingStateMachine(car, view, CFG)

arr.start()

try:
    while fsm.state not in ("DONE", "ABORT"):
        view.poll()
        fsm.update()

        clear_output(wait=True)
        print(fsm.report())

        time.sleep(CFG["LOOP_INTERVAL"])

    clear_output(wait=True)
    print(fsm.report())

except KeyboardInterrupt:
    print("\n사용자 중단")
finally:
    car.all_off()
    arr.stop()
    print("\n모든 모터 정지")

## C-2. 상태 전환 로그

방금 주행에서 각 상태가 언제, 어떤 센서값에서 전환됐는지 확인합니다.
임계값을 어디서 고쳐야 할지 판단하는 근거가 됩니다.

In [ ]:
print(f"{'t(s)':>6}  {'from':14s} -> {'to':14s}   S0     S2     S8     S9    S11")
print("-" * 78)
for e in fsm.log:
    s = e["sensors"]
    vals = "".join(
        f"{s[n]:6.1f} " if s[n] is not None else "  ---- "
        for n in ("S0", "S2", "S8", "S9", "S11")
    )
    print(f"{e['t']:6.2f}  {e['from']:14s} -> {e['to']:14s} {vals}")

if fsm.abort_reason:
    print(f"\n중단 사유: {fsm.abort_reason}")

## C-3. 비상 정지

셀을 강제로 끊는 등의 이유로 모터가 계속 돌고 있을 때 실행하세요.
파트 A만 실행돼 있으면 언제든 동작합니다.

In [ ]:
motor_all_off()
try:
    sonic.write(SONIC_REG_CONTROL, 0x0)
except NameError:
    pass
print("모든 모터 정지 / 초음파 측정 중지")

---

# 현장 튜닝 체크리스트

### 순서
1. **B-1** — 모터 6개가 이름대로 도는지 확인. 다르면 A-2의 `MOTOR_BASE` 수정
2. **B-2** — 핸들 좌우 끝까지 돌려 ADC 범위 확인 → `ADC_MOST_LEFT` / `ADC_MOST_RIGHT`
3. **B-3** — 초음파 5채널이 다 살아있는지 확인
4. **B-5** — 최소 구동 듀티 → `SPEED_ALIGN` / `SPEED_FORWARD` / `SPEED_REVERSE`
5. **B-6** — 조향 도달 시간 → `STEER_SETTLE_TIMEOUT`
6. **B-4** — 장애물 볼 때 / 빈 공간 볼 때 거리 → `BAND_EDGES_CM`
7. **C-1** — 차량을 들어올린 채로 상태 전환만 먼저 확인
8. 바닥에 내려놓고 단계별로 실주행

### 증상별 대처

| 증상 | 고칠 값 |
|---|---|
| 주차 공간을 못 찾고 지나침 | `BAND_EDGES_CM` 상단 경계, `DETECT_BAND` |
| 공간을 찾았는데 너무 일찍/늦게 멈춤 | S0·S2 부착 위치, `CLEAR_BAND` |
| 정렬 중 오차가 줄지 않고 커짐 | **`ALIGN_SIGN`을 -1로 뒤집기** |
| 정렬이 끝나기 전에 공간 깊이가 부족 | `STEER_ALIGN_STEP` 키우기 (2 → 3, 4) |
| 정렬이 좌우로 계속 헌팅 | `ALIGN_DEADBAND_CM` 키우기, `STEER_ALIGN_STEP` 줄이기 |
| 중앙 정렬이 안 됨 | `K_CENTER` 키우기, `OFFSET_MAX_CM` 키우기 |
| 각도는 맞는데 한쪽으로 치우침 | `K_CENTER` 키우기 |
| 후진이 너무 깊게 들어감 | `REAR_STOP_CM` 키우기, S9·S11 부착 각도 확인 |
| 출차 시 우회전이 부족/과함 | `EXIT_TURN_SECONDS` |
| OUT 라인에 못 미침 | `EXIT_STRAIGHT_SECONDS` 늘리기 |

### 배터리 주의

`EXIT_TURN_SECONDS`는 시간 기반이라 배터리 전압에 민감합니다.
**경기 직전 배터리 상태에서** 캘리브레이션하세요.

### 경기 당일

규정상 `.ipynb`가 아닌 `.py`만 사용 가능합니다.
**파트 A 전체**를 하나의 `.py`로 합치고 (단 A-9 정의 확인 셀은 제외),
마지막에 **C-1의 실행 루프**를 붙이면 됩니다. 파트 B는 전부 뺍니다.